In [ ]:
# # 04 — Baseline models

# ## Objective

# Forecast forward EURUSD mid-price changes over 1, 5, 30, and 60 seconds.

# Compare:

# 1. Zero predicted change.
# 2. Continuation of the preceding same-horizon mid-price change.
# 3. Ridge using price and time features.
# 4. Ridge using those features plus displayed-size features.

# ## Development scope

# Use the last three active UTC dates in the training period and the first
# two active UTC dates in the validation period.

# Dates are selected using current-price coverage, without inspecting labels.

# The pilot holdout is not loaded or evaluated.

# ## Fair comparison

# All four forecasts are evaluated on identical timestamps for each horizon.
# Both Ridge models also use identical training observations.

# Price-only eligibility and size-data coverage are reported separately.

# Missing trailing features are imputed using training medians. Missingness
# indicators and scaling are also learned from training data only.

# ## Modeling assumptions

# - Ridge uses one fixed penalty, alpha = 10, without parameter search.
# - Its loss emphasizes squared prediction errors.
# - Labels and forecasts are measured in pips.
# - The trailing-return benchmark assumes continuation, without fitting a
#   multiplier.
# - No computation latency, execution, transaction-cost, or P&L assumptions
#   are introduced.

# ## Interpretation limits

# This is a small development pilot. Its point estimates are descriptive.

# Overlapping labels and serial dependence mean rows are not independent.
# Statistical uncertainty estimates are deferred to the broader walk-forward
# evaluation. Hourly diagnostics here are not confidence intervals.

# Predictive accuracy does not establish achievable trading profits.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(
    (candidate.resolve() for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "pyproject.toml").is_file()
     and (candidate / "src/eurusd_research/config.py").is_file()),
    None,
)
if REPO_ROOT is None:
    REPO_ROOT = Path(input("Full repository folder path: ").strip().strip('"')).expanduser().resolve()

required_files = [
    "pyproject.toml",
    "configs/research_config.example.json",
    "src/eurusd_research/config.py",
]

missing = [
    name for name in required_files
    if not (REPO_ROOT / name).is_file()
]

if missing:
    raise RuntimeError(
        f"Incorrect or incomplete repository folder: {REPO_ROOT}\n"
        f"Missing: {missing}"
    )

os.chdir(REPO_ROOT)
PROJECT_ROOT = REPO_ROOT

from eurusd_research.config import load_config
config = load_config(REPO_ROOT)

print("Repository:", REPO_ROOT)
print("Working directory:", Path.cwd())
print("Configuration loaded:", isinstance(config, dict))

In [ ]:
import subprocess
import sys
from pathlib import Path

folder = input(
    "Full repository folder containing pyproject.toml, configs, and src: "
).strip().strip('"')

if not folder:
    raise ValueError("Enter the repository folder path.")

project_root = Path(folder).expanduser().resolve()
PROJECT_ROOT = project_root
REPO_ROOT = project_root

required_files = [
    "pyproject.toml",
    "configs/research_config.example.json",
    "src/eurusd_research/__init__.py",
    "src/eurusd_research/config.py",
    "src/eurusd_research/features.py",
    "src/eurusd_research/labels.py",
    "src/eurusd_research/validation.py",
]

missing = [
    relative
    for relative in required_files
    if not (project_root / relative).is_file()
]

print("Kernel Python:", sys.executable)
print("Selected repository:", project_root)

if missing:
    raise RuntimeError(
        "The selected folder is incomplete or incorrect. Missing:\n"
        + "\n".join(missing)
        + "\nStop and return this output."
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{project_root}[modeling]",
    ],
    cwd=project_root,
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-c",
        "from pathlib import Path; "
        "import eurusd_research; "
        "from eurusd_research.config import load_config; "
        "print('Package location:', Path(eurusd_research.__file__).resolve())",
    ],
    cwd=project_root,
    check=True,
)

print("SUCCESS: restart the kernel, then resume Notebook 04 from Cell 2.")

In [ ]:
print("Repository:", REPO_ROOT)
print("Configuration loaded:", isinstance(config, dict))

In [ ]:
import subprocess
import sys
from pathlib import Path

project_root = Path(REPO_ROOT).resolve()

required_files = [
    "pyproject.toml",
    "configs/research_config.example.json",
    "src/eurusd_research/__init__.py",
    "src/eurusd_research/config.py",
    "src/eurusd_research/features.py",
    "src/eurusd_research/labels.py",
    "src/eurusd_research/validation.py",
]

missing = [
    relative
    for relative in required_files
    if not (project_root / relative).is_file()
]

print("Kernel Python:", sys.executable)
print("Installing project from:", project_root)

if missing:
    raise RuntimeError(
        "This project folder is incomplete. Missing: "
        + ", ".join(missing)
        + ". Stop and return this output."
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{project_root}[modeling]",
    ],
    cwd=project_root,
    check=True,
)

# Verify the installation in a fresh Python process.
subprocess.run(
    [
        sys.executable,
        "-c",
        "from pathlib import Path; "
        "import eurusd_research; "
        "from eurusd_research.config import load_config; "
        "print('Package location:', Path(eurusd_research.__file__).resolve())",
    ],
    cwd=project_root / "notebooks",
    check=True,
)

print("Setup completed. Restart the kernel, then rerun Cell 2 onward.")

In [ ]:
import hashlib
import importlib
import json
import platform
import shutil
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK = "04_baseline_models.ipynb"
notebook_started = time.perf_counter()

# Leave None when Jupyter is running inside the repository.
# Otherwise enter the repository root, using Path(r"C:\...").
PROJECT_ROOT = None

if PROJECT_ROOT is None:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    REPO_ROOT = next(
        (
            path for path in candidates
            if (path / "configs/research_config.example.json").is_file()
            and (path / "src/eurusd_research/features.py").is_file()
        ),
        None,
    )
else:
    REPO_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

if REPO_ROOT is None or not (
    REPO_ROOT / "configs/research_config.example.json"
).is_file():
    raise RuntimeError(
        "Set PROJECT_ROOT to the repository containing configs/ and src/."
    )

from eurusd_research.config import load_config
from eurusd_research import features as feature_io
from eurusd_research.data import _sha256
from eurusd_research.validation import utc, validate_plan

config = load_config(REPO_ROOT)
feature_settings = config["sampling_features"]
HORIZONS = config["labels_validation"]["horizons_seconds"]

if HORIZONS != [1, 5, 30, 60]:
    raise RuntimeError("Expected horizons [1, 5, 30, 60].")
if feature_settings["max_staleness_seconds"] != 30:
    raise RuntimeError("Expected the confirmed 30-second snapshot policy.")

feature_manifest_path = (
    REPO_ROOT / config["active_features_manifest"]
).resolve()
label_manifest_path = (
    REPO_ROOT / config["active_labels_manifest"]
).resolve()

feature_state = json.loads(
    feature_manifest_path.read_text(encoding="utf-8")
)
label_state = json.loads(
    label_manifest_path.read_text(encoding="utf-8")
)
plan = validate_plan(config["pilot_validation_plan"])

if feature_state["status"] != "complete" or label_state["status"] != "complete":
    raise RuntimeError("Completed feature and label generations are required.")
if feature_state["job_id"] != plan["feature_job_id"]:
    raise RuntimeError("The feature generation and split plan differ.")
if label_state["identity"]["plan"] != plan:
    raise RuntimeError("The labels and current split plan differ.")
if (
    label_state["identity"]["input_manifest_sha256"][0]
    != _sha256(feature_manifest_path)
):
    raise RuntimeError("The active features differ from those used for labels.")
if feature_state["identity"]["settings"] != feature_settings:
    raise RuntimeError("Feature configuration differs from the saved generation.")

default_model_settings = {
    "training_active_dates": 3,
    "validation_active_dates": 2,
    "max_rows_per_split": 300000,
    "max_frame_mib": 192,
    "max_combined_frame_mib": 256,
    "minimum_fit_rows": 1000,
    "ridge_alpha": 10.0,
    "forecast_buckets": 5,
    "price_features": [
        "spread_pips",
        "mid_change_1s_pips",
        "mid_change_5s_pips",
        "mid_change_30s_pips",
        "mid_change_60s_pips",
        "realized_vol_60s_pips",
        "realized_vol_300s_pips",
        "spread_change_1s_pips",
        "spread_change_5s_pips",
        "snapshot_staleness_seconds",
        "bid_price_change_age_seconds",
        "ask_price_change_age_seconds",
        "utc_time_sin",
        "utc_time_cos",
        "london_time_sin",
        "london_time_cos",
        "new_york_time_sin",
        "new_york_time_cos",
    ],
    "size_features": [
        "bid_size",
        "ask_size",
        "size_imbalance",
        "size_weighted_minus_mid_pips",
        "bid_size_change_1s",
        "ask_size_change_1s",
        "bid_size_change_5s",
        "ask_size_change_5s",
    ],
    "required_current_size_features": [
        "bid_size",
        "ask_size",
        "size_imbalance",
        "size_weighted_minus_mid_pips",
    ],
}

model_settings = config.get("baseline_models", default_model_settings)

try:
    sklearn_version = version("scikit-learn")
except PackageNotFoundError:
    sklearn_version = None

new_sources = {}

print("NOTEBOOK_04_PREFLIGHT")
print("Python:", platform.python_version())
print("Repository:", REPO_ROOT)
print("Feature generation:", feature_state["job_id"])
print("Label generation:", label_state["job_id"])
print("scikit-learn:", sklearn_version or "NOT INSTALLED")
print("Ridge alpha:", model_settings["ridge_alpha"])
print("Pilot holdout will not be loaded.")

In [ ]:
# ## Optional modeling dependency installation

# The authoritative dependency definition remains pyproject.toml.

# Its existing modeling extra contains scikit-learn. Installing that extra
# also resolves scikit-learn's required dependencies.

# The following cell does nothing unless RUN_MODELING_INSTALL is explicitly
# set to True. It uses this kernel's Python executable and does not request
# a general package upgrade.

# If installation runs:

# 1. Restart the Jupyter kernel immediately afterward.
# 2. Set RUN_MODELING_INSTALL back to False.
# 3. Resume from Cell 2 and execute sequentially.

# If scikit-learn is already installed and imports correctly, leave the flag
# False and continue.

In [ ]:
RUN_MODELING_INSTALL = False

if RUN_MODELING_INSTALL:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "-e", f"{REPO_ROOT}[modeling]",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
    print(
        "Installation completed. Restart the kernel now, set this flag "
        "back to False, and resume from Cell 2."
    )
elif sklearn_version is None:
    raise RuntimeError(
        "scikit-learn is missing. Set RUN_MODELING_INSTALL=True, run this "
        "cell, restart the kernel, restore False, and resume from Cell 2."
    )
else:
    print("Installation skipped; using the installed modeling environment.")

In [ ]:
# ## Modeling pipeline

# Each Ridge model contains:

# 1. A training-median imputer with missingness indicators.
# 2. A scaler fitted to the imputed training features.
# 3. Ridge regression.

# An entirely missing training feature is retained with a zero replacement.
# It cannot acquire useful variation from unseen validation values during
# training.

# The price-only model excludes explicit displayed-size values. Both models
# still use the same source feed and sampling policy; this comparison
# measures the incremental contribution of the added size features.

# Only explicit predictor lists are accepted. Targets, label diagnostics,
# split fields, and source identifiers cannot enter those lists.

In [ ]:
import subprocess
import sys

check = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "from sklearn.impute import SimpleImputer; "
            "from sklearn.preprocessing import StandardScaler; "
            "from sklearn.linear_model import Ridge; "
            "print('PASS: modeling imports work in a fresh Python process.')"
        ),
    ],
    capture_output=True,
    text=True,
    errors="replace",
    timeout=60,
)

print("FRESH_PROCESS_IMPORT_CHECK")
print("Python:", sys.executable)
print("Exit code:", check.returncode)
print(check.stdout.strip())

if check.stderr.strip():
    print(check.stderr.strip())

In [ ]:
if RUN_MODELING_INSTALL:
    raise RuntimeError(
        "Restart the kernel after installation, restore the flag to False, "
        "and resume from Cell 2."
    )

import inspect
import sklearn
from sklearn.impute import SimpleImputer

if "keep_empty_features" not in inspect.signature(SimpleImputer).parameters:
    raise RuntimeError(
        "This scikit-learn version lacks keep_empty_features. "
        "Return this message and the installed version before continuing."
    )

versions = {
    name: version(name)
    for name in (
        "numpy", "pandas", "pyarrow", "duckdb", "scikit-learn",
        "scipy", "joblib", "pytest", "matplotlib",
    )
}

print("MODELING_ENVIRONMENT")
print(json.dumps(versions, indent=2))

for relative in (
    "src/eurusd_research/models.py",
    "src/eurusd_research/evaluation.py",
    "tests/test_models.py",
    "tests/test_evaluation.py",
):
    print(relative, "EXISTS" if (REPO_ROOT / relative).exists() else "MISSING")

In [ ]:
# ## Reusable modeling functions

# models.py will provide:

# - bounded loading of selected development periods;
# - selected-file integrity checks;
# - explicit eligibility masks;
# - training-only Ridge fitting;
# - prediction and atomic model saving.

# Only Parquet parts intersecting the requested development window are
# opened. Holdout parts are excluded from the selected file lists.

# The loader also reconciles current-price and source-row provenance
# between features and labels.

In [ ]:
new_sources["src/eurusd_research/models.py"] = r'''
"""Bounded development samples and training-only Ridge pipelines."""

import os
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from . import features as io
from .data import _sha256
from .validation import utc


def validate_predictors(columns):
    forbidden = (
        "target_", "label_", "split_", "same_source_",
        "source_", "reference_", "current_", "feature_timestamp",
    )
    if not columns or len(columns) != len(set(columns)):
        raise ValueError("Predictor names must be nonempty and unique.")
    for name in columns:
        if not isinstance(name, str) or not name.isidentifier():
            raise ValueError("Predictor names must be valid identifiers.")
        if name.startswith(forbidden):
            raise ValueError(f"Forbidden predictor: {name}")
    return list(columns)


def _selected_files(state, start, end):
    if state["status"] != "complete":
        raise ValueError("A completed dataset is required.")
    root = Path(state["output_dir"]).resolve()
    files = []
    for part in state["parts"]:
        if utc(part["start_utc"]) >= end or utc(part["end_utc"]) <= start:
            continue
        path = (root / part["path"]).resolve()
        if not path.is_relative_to(root):
            raise RuntimeError("A selected output escapes its generation directory.")
        if not path.is_file() or _sha256(path) != part["sha256"]:
            raise RuntimeError(f"Selected output integrity failed: {path}")
        parquet = pq.ParquetFile(path)
        if (
            parquet.metadata.num_rows != part["rows"]
            or io._schema_hash(parquet.schema_arrow) != part["schema_sha256"]
        ):
            raise RuntimeError("Selected output rows or schema changed.")
        files.append(str(path))
    if not files:
        raise ValueError("No dataset parts intersect the requested window.")
    return files


def load_development_window(feature_state, label_state, start, end, columns,
                            horizons, resource_settings, maximum_rows,
                            maximum_frame_mib, working_directory):
    start, end = utc(start), utc(end)
    columns = validate_predictors(columns)
    expected = int((end - start).total_seconds())
    if expected < 1 or expected > maximum_rows:
        raise ValueError("The development window exceeds its clock-row limit.")
    if any(type(h) is not int or h < 1 for h in horizons):
        raise ValueError("Horizons must be positive integer seconds.")

    feature_files = _selected_files(feature_state, start, end)
    label_files = _selected_files(label_state, start, end)
    connection = io._connect(
        feature_files, "features", working_directory, resource_settings
    )
    try:
        connection.read_parquet(
            label_files, hive_partitioning=False
        ).create_view("labels")
        selected = [
            "f.feature_timestamp_utc",
            "f.source_timestamp_utc",
            "f.source_row",
            "f.price_valid",
            "f.size_pair_valid",
            "f.mid_price AS current_mid_price",
            "l.mid_price AS reference_mid_price",
            "l.source_row AS reference_source_row",
        ]
        selected += [f"f.{name}" for name in columns]
        for h in horizons:
            selected += [
                f"l.target_timestamp_utc_{h}s",
                f"l.label_{h}s_pips",
                f"l.label_valid_{h}s",
                f"l.same_source_record_{h}s",
                f"l.split_{h}s",
            ]
        frame = connection.execute(f"""
            SELECT {", ".join(selected)}
            FROM features f
            JOIN labels l USING (feature_timestamp_utc)
            WHERE f.feature_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
              AND f.feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
            ORDER BY f.feature_timestamp_utc
            LIMIT ?
        """, [start.isoformat(), end.isoformat(), maximum_rows + 1]).fetchdf()
    finally:
        connection.close()

    for name in frame:
        if "timestamp_utc" in name:
            frame[name] = pd.to_datetime(frame[name], utc=True).astype(
                "datetime64[us, UTC]"
            )
        elif name.startswith("split_"):
            frame[name] = frame[name].astype("category")

    times = frame["feature_timestamp_utc"]
    if (
        len(frame) != expected or not times.is_unique
        or times.iloc[0] != start
        or times.iloc[-1] != end - pd.Timedelta(seconds=1)
        or not times.diff().iloc[1:].eq(pd.Timedelta(seconds=1)).all()
    ):
        raise RuntimeError("Development grid conservation failed.")
    if (frame["source_timestamp_utc"] > times).any():
        raise RuntimeError("A future source entered development features.")
    if not np.allclose(
        frame["current_mid_price"], frame["reference_mid_price"], equal_nan=True
    ):
        raise RuntimeError("Feature and label current prices disagree.")
    if not frame["source_row"].astype("Int64").equals(
        frame["reference_source_row"].astype("Int64")
    ):
        raise RuntimeError("Feature and label source rows disagree.")

    observed_bytes = int(frame.memory_usage(index=True, deep=True).sum())
    if observed_bytes > maximum_frame_mib * 1024**2:
        raise RuntimeError("The collected frame exceeds its configured memory guard.")
    return frame


def eligibility(frame, horizon, role, window_end, required_size_columns):
    target = frame[f"target_timestamp_utc_{horizon}s"]
    valid = (
        frame["price_valid"]
        & frame[f"label_valid_{horizon}s"]
        & np.isfinite(frame[f"label_{horizon}s_pips"])
        & frame[f"split_{horizon}s"].eq(role)
    )
    inside_window = target.lt(utc(window_end))
    benchmark_available = np.isfinite(frame[f"mid_change_{horizon}s_pips"])
    price = valid & inside_window & benchmark_available

    size_values = frame[required_size_columns].to_numpy(dtype=float)
    size_available = (
        frame["size_pair_valid"] & np.isfinite(size_values).all(axis=1)
    )
    common = price & size_available

    counts = {
        "clock_rows": len(frame),
        "price_valid_rows": int(frame["price_valid"].sum()),
        "valid_label_rows": int(frame[f"label_valid_{horizon}s"].sum()),
        "role_eligible_before_window_purge": int(valid.sum()),
        "additional_window_boundary_purges": int((valid & ~inside_window).sum()),
        "price_eligible_rows": int(price.sum()),
        "matched_rows": int(common.sum()),
        "price_rows_without_required_size": int((price & ~size_available).sum()),
        "price_rows_missing_trailing_benchmark": int(
            (valid & inside_window & ~benchmark_available).sum()
        ),
    }
    return price, common, counts


def numeric_predictors(frame, columns):
    columns = validate_predictors(columns)
    return frame.loc[:, columns].astype("float64").replace(
        [np.inf, -np.inf], np.nan
    )


def fit_ridge(frame, y, columns, alpha=10.0, minimum_rows=1000):
    columns = validate_predictors(columns)
    y = np.asarray(y, dtype=float)
    if (
        y.ndim != 1 or len(y) != len(frame)
        or len(y) < minimum_rows or not np.isfinite(y).all()
    ):
        raise ValueError("Training targets must be finite and satisfy the row minimum.")
    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError("Ridge alpha must be finite and positive.")

    pipeline = Pipeline([
        ("impute", SimpleImputer(
            strategy="median", add_indicator=True, keep_empty_features=True
        )),
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, solver="cholesky")),
    ])
    pipeline.fit(numeric_predictors(frame, columns), y)
    return pipeline


def predict_ridge(pipeline, frame):
    columns = pipeline.feature_names_in_.tolist()
    result = pipeline.predict(numeric_predictors(frame, columns))
    if not np.isfinite(result).all():
        raise RuntimeError("The model produced nonfinite forecasts.")
    return result


def save_pipeline(pipeline, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, name = tempfile.mkstemp(
        prefix=f".{path.name}.", suffix=".tmp", dir=path.parent
    )
    temporary = Path(name)
    try:
        with os.fdopen(descriptor, "wb") as handle:
            joblib.dump(pipeline, handle, compress=3)
            handle.flush()
            os.fsync(handle.fileno())
        io._replace_path(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    return {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": _sha256(path),
    }
'''.lstrip()

print("Prepared models.py")

In [ ]:
new_sources["src/eurusd_research/evaluation.py"] = r'''
"""Regression diagnostics without model selection or execution assumptions."""

import numpy as np
import pandas as pd


def _arrays(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if (
        actual.ndim != 1 or predicted.ndim != 1
        or len(actual) != len(predicted) or len(actual) == 0
        or not np.isfinite(actual).all()
        or not np.isfinite(predicted).all()
    ):
        raise ValueError("Actual and predicted values must be finite matching vectors.")
    return actual, predicted


def regression_metrics(actual, predicted):
    actual, predicted = _arrays(actual, predicted)
    error = predicted - actual
    slope = intercept = None
    if np.ptp(predicted) > 1e-12:
        centered = predicted - predicted.mean()
        slope = float(
            np.dot(centered, actual - actual.mean())
            / np.dot(centered, centered)
        )
        intercept = float(actual.mean() - slope * predicted.mean())
    return {
        "rows": len(actual),
        "mae_pips": float(np.mean(np.abs(error))),
        "rmse_pips": float(np.sqrt(np.mean(error**2))),
        "mean_actual_pips": float(actual.mean()),
        "mean_forecast_pips": float(predicted.mean()),
        "forecast_bias_pips": float(error.mean()),
        "calibration_slope": slope,
        "calibration_intercept_pips": intercept,
    }


def training_bucket_cuts(predictions, buckets=5):
    predictions = np.asarray(predictions, dtype=float)
    if (
        predictions.ndim != 1 or len(predictions) == 0
        or not np.isfinite(predictions).all()
        or type(buckets) is not int or buckets < 2
    ):
        raise ValueError("Finite training predictions and at least two buckets required.")
    cuts = np.unique(np.quantile(
        predictions, np.linspace(0, 1, buckets + 1)[1:-1]
    ))
    return cuts[
        (cuts > predictions.min()) & (cuts < predictions.max())
    ]


def forecast_buckets(actual, predicted, cuts):
    actual, predicted = _arrays(actual, predicted)
    cuts = np.asarray(cuts, dtype=float)
    if (
        not np.isfinite(cuts).all()
        or (len(cuts) > 1 and not np.all(np.diff(cuts) > 0))
    ):
        raise ValueError("Bucket cuts must be finite and strictly increasing.")
    frame = pd.DataFrame({
        "bucket": np.searchsorted(cuts, predicted, side="right"),
        "forecast_pips": predicted,
        "actual_pips": actual,
    })
    return frame.groupby("bucket", as_index=False).agg(
        rows=("actual_pips", "size"),
        mean_forecast_pips=("forecast_pips", "mean"),
        mean_actual_pips=("actual_pips", "mean"),
    )


def hourly_errors(timestamps, actual, predicted):
    actual, predicted = _arrays(actual, predicted)
    times = pd.Series(pd.to_datetime(timestamps, utc=True)).reset_index(drop=True)
    if len(times) != len(actual) or times.isna().any():
        raise ValueError("Valid matching timestamps are required.")
    error = predicted - actual
    frame = pd.DataFrame({
        "hour_utc": times.dt.floor("h"),
        "absolute_error": np.abs(error),
        "squared_error": error**2,
    })
    result = frame.groupby("hour_utc", as_index=False).agg(
        rows=("absolute_error", "size"),
        mae_pips=("absolute_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
    )
    result["rmse_pips"] = np.sqrt(result.pop("mean_squared_error"))
    return result
'''.lstrip()

print("Prepared evaluation.py")

In [ ]:
new_sources["tests/test_models.py"] = r'''
import joblib
import numpy as np
import pandas as pd
import pytest

from eurusd_research import features as io
from eurusd_research.models import (
    eligibility, fit_ridge, load_development_window,
    predict_ridge, save_pipeline, validate_predictors,
)


def test_outcome_predictors_rejected():
    for name in (
        "target_mid_price_1s", "label_1s_pips",
        "split_1s", "same_source_record_1s", "source_row",
    ):
        with pytest.raises(ValueError, match="Forbidden"):
            validate_predictors(["spread_pips", name])


def test_preprocessing_is_fitted_on_training_only():
    train = pd.DataFrame({"x": [1.0, 3.0, np.nan, 5.0]})
    model = fit_ridge(train, [0, 1, 0, 1], ["x"], minimum_rows=2)
    imputer_before = model.named_steps["impute"].statistics_.copy()
    scaler_before = model.named_steps["scale"].mean_.copy()

    predict_ridge(model, pd.DataFrame({"x": [1000000.0, np.nan]}))

    np.testing.assert_array_equal(
        model.named_steps["impute"].statistics_, imputer_before
    )
    np.testing.assert_array_equal(
        model.named_steps["scale"].mean_, scaler_before
    )
    assert imputer_before[0] == 3.0


def test_all_missing_training_feature_is_supported():
    train = pd.DataFrame({"x": [1.0, 2.0, 3.0], "empty": [np.nan] * 3})
    model = fit_ridge(train, [0, 1, 2], ["x", "empty"], minimum_rows=2)
    result = predict_ridge(
        model, pd.DataFrame({"x": [2.0], "empty": [999.0]})
    )
    assert np.isfinite(result).all()


def test_eligibility_keeps_price_and_size_coverage_separate():
    times = pd.date_range("2020-01-01", periods=2, freq="s", tz="UTC")
    frame = pd.DataFrame({
        "price_valid": [True, True],
        "size_pair_valid": [True, False],
        "target_timestamp_utc_1s": times + pd.Timedelta(seconds=1),
        "label_1s_pips": [0.0, 1.0],
        "label_valid_1s": [True, True],
        "split_1s": ["train", "train"],
        "mid_change_1s_pips": [0.0, 0.0],
        "bid_size": [1.0, np.nan],
    })
    price, common, counts = eligibility(
        frame, 1, "train", times[-1] + pd.Timedelta(seconds=2), ["bid_size"]
    )
    assert price.tolist() == [True, True]
    assert common.tolist() == [True, False]
    assert counts["price_rows_without_required_size"] == 1

    price, _, counts = eligibility(
        frame, 1, "train", times[-1] + pd.Timedelta(seconds=1), ["bid_size"]
    )
    assert price.tolist() == [True, False]
    assert counts["additional_window_boundary_purges"] == 1


def test_saved_pipeline_preserves_predictions(tmp_path):
    train = pd.DataFrame({"x": [1.0, 2.0, 3.0]})
    model = fit_ridge(train, [0, 1, 2], ["x"], minimum_rows=2)
    path = tmp_path / "own_model.joblib"
    save_pipeline(model, path)
    restored = joblib.load(path)
    np.testing.assert_allclose(
        predict_ridge(model, train), predict_ridge(restored, train)
    )


def test_loader_does_not_open_holdout_parts(tmp_path):
    times = pd.date_range("2020-01-01", periods=2, freq="s", tz="UTC")
    features = pd.DataFrame({
        "feature_timestamp_utc": times,
        "source_timestamp_utc": times,
        "source_row": [1, 2],
        "price_valid": [True, True],
        "size_pair_valid": [True, True],
        "mid_price": [1.1, 1.1001],
        "spread_pips": [1.0, 1.0],
    })
    labels = pd.DataFrame({
        "feature_timestamp_utc": times,
        "source_row": [1, 2],
        "mid_price": [1.1, 1.1001],
        "target_timestamp_utc_1s": times + pd.Timedelta(seconds=1),
        "label_1s_pips": [1.0, 0.0],
        "label_valid_1s": [True, True],
        "same_source_record_1s": [False, False],
        "split_1s": ["train", "train"],
    })
    start, end = times[0], times[-1] + pd.Timedelta(seconds=1)

    def dataset(frame, name):
        folder = tmp_path / name
        record = io._save_frame(frame, folder / "part.parquet")
        record.update({
            "path": "part.parquet",
            "start_utc": start.isoformat(), "end_utc": end.isoformat(),
        })
        return {
            "status": "complete", "output_dir": str(folder),
            "parts": [
                record,
                {
                    "path": "holdout_file_must_not_be_opened.parquet",
                    "start_utc": end.isoformat(),
                    "end_utc": (end + pd.Timedelta(seconds=1)).isoformat(),
                },
            ],
        }

    result = load_development_window(
        dataset(features, "features"), dataset(labels, "labels"),
        start, end, ["spread_pips"], [1],
        {"duckdb_memory_limit": "128MB", "duckdb_temp_limit": "1GB"},
        10, 10, tmp_path / "working",
    )
    assert len(result) == 2
'''.lstrip()

new_sources["tests/test_evaluation.py"] = r'''
import numpy as np
import pandas as pd
import pytest

from eurusd_research.evaluation import (
    forecast_buckets, hourly_errors, regression_metrics, training_bucket_cuts,
)


def test_hand_calculated_errors():
    metrics = regression_metrics([1.0, -1.0], [0.0, 0.0])
    assert metrics["mae_pips"] == 1.0
    assert metrics["rmse_pips"] == 1.0
    assert metrics["calibration_slope"] is None


def test_calibration_on_known_linear_relation():
    metrics = regression_metrics([1.0, 3.0, 5.0], [0.0, 1.0, 2.0])
    assert metrics["calibration_slope"] == pytest.approx(2.0)
    assert metrics["calibration_intercept_pips"] == pytest.approx(1.0)


def test_buckets_use_training_boundaries():
    cuts = training_bucket_cuts([0.0, 1.0, 2.0, 3.0], buckets=2)
    np.testing.assert_allclose(cuts, [1.5])
    table = forecast_buckets([0.0, 0.0], [-100.0, 100.0], cuts)
    assert table["bucket"].tolist() == [0, 1]
    assert table["rows"].sum() == 2
    assert len(training_bucket_cuts([0.0, 0.0, 0.0])) == 0


def test_hourly_aggregation_preserves_error_totals():
    times = pd.to_datetime([
        "2020-01-01T00:00:00Z",
        "2020-01-01T00:00:01Z",
        "2020-01-01T01:00:00Z",
    ], format="ISO8601", utc=True)
    table = hourly_errors(times, [1.0, -1.0, 2.0], [0.0, 0.0, 0.0])
    assert table["rows"].tolist() == [2, 1]
    assert table["mae_pips"].tolist() == [1.0, 2.0]
    assert (table["rows"] * table["mae_pips"]).sum() == 4.0
'''.lstrip()

print("Prepared modeling and evaluation tests")

In [ ]:
for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if path.exists():
        existing = path.read_text(encoding="utf-8").replace("\r\n", "\n")
        if existing != source:
            raise RuntimeError(
                f"Existing contents differ: {relative}. "
                "Stop; no existing source file has been overwritten."
            )

for relative, source in new_sources.items():
    path = REPO_ROOT / relative
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(source)
        print("CREATED", relative)
    else:
        print("REUSED", relative)

importlib.invalidate_caches()

probe = subprocess.run(
    [
        sys.executable, "-c",
        "import json; from pathlib import Path; "
        "from eurusd_research import models, evaluation; "
        "print(json.dumps({name: str(Path(module.__file__).resolve()) "
        "for name, module in [('models', models), ('evaluation', evaluation)]}))",
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True,
    text=True,
)
if probe.returncode:
    print(probe.stdout)
    print(probe.stderr)
    raise RuntimeError("Project imports from notebooks/ failed.")

for name, filename in json.loads(probe.stdout).items():
    expected = REPO_ROOT / "src/eurusd_research" / f"{name}.py"
    if Path(filename).resolve() != expected.resolve():
        raise RuntimeError(f"Imported the wrong module: {name}")

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode:
    raise RuntimeError("Tests failed. Stop and return Cell 14 output.")

test_summary = test_run.stdout.strip().splitlines()[-1]

from eurusd_research.models import (
    eligibility, fit_ridge, load_development_window,
    numeric_predictors, predict_ridge, save_pipeline, validate_predictors,
)
from eurusd_research.evaluation import (
    forecast_buckets, hourly_errors, regression_metrics, training_bucket_cuts,
)

PRICE_FEATURES = validate_predictors(model_settings["price_features"])
SIZE_FEATURES = validate_predictors(model_settings["size_features"])
ALL_FEATURES = validate_predictors(PRICE_FEATURES + SIZE_FEATURES)
SIZE_REQUIRED = model_settings["required_current_size_features"]

if not set(SIZE_REQUIRED).issubset(SIZE_FEATURES):
    raise RuntimeError("Required current size features must belong to the size family.")

for relative in (
    "artifacts/models/notebook04/example.joblib",
    "artifacts/forecasts/notebook04/example.parquet",
    "artifacts/manifests/baselines/example.json",
    "reports/private/notebook04_execution.json",
    "reports/tables/notebook04/example.csv",
    "reports/figures/notebook04/example.png",
    "configs/research_config.local.json",
):
    check = subprocess.run(
        ["git", "check-ignore", "--no-index", "-q", "--", relative],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    if check.returncode != 0:
        raise RuntimeError(f"Generated/private output is not safely ignored: {relative}")

print("PASS: imports, correctness tests, predictor lists, and output ignore rules.")

In [ ]:
# Resolve generated-data locations relative to the current repository.
# Saved manifests and their checksums remain unchanged.

def load_relocated_generation(manifest_path, dataset_name):
    state = json.loads(
        Path(manifest_path).read_text(encoding="utf-8")
    )
    if state["status"] != "complete":
        raise RuntimeError(f"{dataset_name} generation is incomplete.")

    output_dir = (
        REPO_ROOT / "data/processed" / dataset_name / state["job_id"]
    ).resolve()

    for part in state["parts"]:
        path = (output_dir / part["path"]).resolve()

        if not path.is_relative_to(output_dir):
            raise RuntimeError("A partition path escapes its generation folder.")
        if not path.is_file():
            raise RuntimeError(f"Relocated data file is missing:\n{path}")
        if path.stat().st_size != part["bytes"]:
            raise RuntimeError(f"Relocated data file size differs:\n{path}")

    # Change only the runtime copy, not the saved manifest.
    relocated = dict(state)
    relocated["output_dir"] = str(output_dir)

    print(f"{dataset_name}: located {len(state['parts'])} partition files.")
    return relocated


feature_state = load_relocated_generation(
    feature_manifest_path, "features"
)
label_state = load_relocated_generation(
    label_manifest_path, "labels"
)

if (
    label_state["identity"]["input_manifest_sha256"][0]
    != _sha256(feature_manifest_path)
):
    raise RuntimeError("The feature-to-label manifest checksum link differs.")

periods = {item["role"]: item for item in plan["periods"]}
holdout_start = utc(periods["pilot_holdout"]["start_utc"])

development_parts = [
    part for part in feature_state["parts"]
    if utc(part["start_utc"]) < holdout_start
]
development_files = [
    str(Path(feature_state["output_dir"]) / part["path"])
    for part in development_parts
]

connection = feature_io._connect(
    development_files,
    "features",
    REPO_ROOT / "artifacts/manifests/baselines/planning",
    feature_settings,
)
try:
    coverage_by_date = connection.execute("""
        SELECT date_trunc('day', feature_timestamp_utc) AS date_utc,
               count(*) FILTER (WHERE price_valid) AS usable_price_rows
        FROM features
        WHERE feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
        GROUP BY 1 ORDER BY 1
    """, [holdout_start.isoformat()]).fetchdf()
finally:
    connection.close()

coverage_by_date["date_utc"] = pd.to_datetime(
    coverage_by_date["date_utc"], utc=True
)

windows = {}
for role, number, choose_last in (
    ("train", model_settings["training_active_dates"], True),
    ("validation", model_settings["validation_active_dates"], False),
):
    period = periods[role]
    period_start = utc(period["start_utc"])
    period_end = utc(period["end_utc"])

    available = coverage_by_date.loc[
        coverage_by_date["usable_price_rows"].gt(0)
        & coverage_by_date["date_utc"].ge(period_start)
        & coverage_by_date["date_utc"].lt(period_end),
        "date_utc",
    ].tolist()

    if type(number) is not int or number < 1 or len(available) < number:
        raise RuntimeError(f"Insufficient active dates for {role}.")

    selected = available[-number:] if choose_last else available[:number]
    start = max(period_start, selected[0])
    end = min(period_end, selected[-1] + pd.Timedelta(days=1))

    if int((end - start).total_seconds()) > model_settings["max_rows_per_split"]:
        raise RuntimeError(
            f"{role} exceeds the configured row guard. "
            "Use fewer active dates before collecting data."
        )

    windows[role] = {
        "start_utc": start.isoformat(),
        "end_utc": end.isoformat(),
        "selected_active_dates": [
            date.date().isoformat() for date in selected
        ],
    }

if utc(windows["train"]["end_utc"]) > utc(windows["validation"]["start_utc"]):
    raise RuntimeError("Development windows overlap.")
if utc(windows["validation"]["end_utc"]) > holdout_start:
    raise RuntimeError("The validation window enters the pilot holdout.")

frames = {}
for role, window in windows.items():
    frames[role] = load_development_window(
        feature_state,
        label_state,
        window["start_utc"],
        window["end_utc"],
        ALL_FEATURES,
        HORIZONS,
        feature_settings,
        model_settings["max_rows_per_split"],
        model_settings["max_frame_mib"],
        REPO_ROOT / "artifacts/manifests/baselines/loading" / role,
    )

frame_bytes = {
    role: int(frame.memory_usage(index=True, deep=True).sum())
    for role, frame in frames.items()
}

if sum(frame_bytes.values()) > (
    model_settings["max_combined_frame_mib"] * 1024**2
):
    raise RuntimeError("Combined development frames exceed the memory guard.")

print("DEVELOPMENT_WINDOWS")
print(json.dumps(windows, indent=2))
print("Collected clock rows:", {
    role: len(frame) for role, frame in frames.items()
})
print("Observed frame bytes:", frame_bytes)
print("Pilot holdout starts:", holdout_start.isoformat())
print("PASS: relocated inputs loaded; no pilot holdout rows loaded.")

In [ ]:
if shutil.disk_usage(REPO_ROOT).free < 2 * 1024**3:
    raise RuntimeError("Free at least 2 GiB before fitting and saving outputs.")

input_hashes = {
    "features": _sha256(feature_manifest_path),
    "labels": _sha256(label_manifest_path),
}
implementation_hashes = {
    name: _sha256(REPO_ROOT / "src/eurusd_research" / f"{name}.py")
    for name in ("models", "evaluation")
}

identity = {
    "feature_job_id": feature_state["job_id"],
    "label_job_id": label_state["job_id"],
    "input_manifest_sha256": input_hashes,
    "windows": windows,
    "settings": model_settings,
    "horizons_seconds": HORIZONS,
    "implementation_sha256": implementation_hashes,
    "python": platform.python_version(),
    "versions": versions,
}

run_id = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode("utf-8")
).hexdigest()[:24]

run_manifest_path = (
    REPO_ROOT / "artifacts/manifests/baselines" / f"job-{run_id}.json"
)
if run_manifest_path.exists():
    previous = json.loads(run_manifest_path.read_text(encoding="utf-8"))
    if previous["identity"] != identity:
        raise RuntimeError("Existing baseline run identity is incompatible.")

model_dir = REPO_ROOT / "artifacts/models" / run_id
forecast_dir = REPO_ROOT / "artifacts/forecasts" / run_id
table_dir = REPO_ROOT / "reports/tables" / run_id
for folder in (model_dir, forecast_dir, table_dir):
    folder.mkdir(parents=True, exist_ok=True)

run_state = {
    "kind": "development_baselines",
    "run_id": run_id,
    "status": "running",
    "identity": identity,
    "completed_horizons": [],
    "files": [],
}
feature_io.write_json(run_manifest_path, run_state)

coverage_records = []
metric_records = []
bucket_tables = []
hourly_tables = []
conditional_records = []
coefficient_tables = []
fit_records = []
bucket_cut_records = {}
missing_feature_records = []

fit_started = time.perf_counter()

for h in HORIZONS:
    train = frames["train"]
    validation = frames["validation"]

    _, train_mask, train_counts = eligibility(
        train, h, "train", windows["train"]["end_utc"], SIZE_REQUIRED
    )
    _, validation_mask, validation_counts = eligibility(
        validation, h, "validation", windows["validation"]["end_utc"], SIZE_REQUIRED
    )

    for role, counts in (("train", train_counts), ("validation", validation_counts)):
        coverage_records.append({
            "horizon_seconds": h, "role": role, **counts,
        })

    if (
        train_mask.sum() < model_settings["minimum_fit_rows"]
        or validation_mask.sum() < model_settings["minimum_fit_rows"]
    ):
        raise RuntimeError(f"Insufficient matched development rows at horizon {h}.")

    training_common = train.loc[train_mask]
    validation_common = validation.loc[validation_mask]
    y_train = training_common[f"label_{h}s_pips"].to_numpy(dtype=float)
    y_validation = validation_common[f"label_{h}s_pips"].to_numpy(dtype=float)

    train_predictions = {
        "zero": np.zeros(len(training_common)),
        "trailing": training_common[f"mid_change_{h}s_pips"].to_numpy(dtype=float),
    }
    validation_predictions = {
        "zero": np.zeros(len(validation_common)),
        "trailing": validation_common[f"mid_change_{h}s_pips"].to_numpy(dtype=float),
    }

    training_row_hash = hashlib.sha256(
        pd.DatetimeIndex(training_common["feature_timestamp_utc"])
        .astype("datetime64[us, UTC]").asi8.tobytes()
    ).hexdigest()
    validation_row_hash = hashlib.sha256(
        pd.DatetimeIndex(validation_common["feature_timestamp_utc"])
        .astype("datetime64[us, UTC]").asi8.tobytes()
    ).hexdigest()

    for family, columns in (
        ("ridge_price", PRICE_FEATURES),
        ("ridge_price_size", ALL_FEATURES),
    ):
        numeric = numeric_predictors(training_common, columns)
        for name in columns:
            missing_feature_records.append({
                "horizon_seconds": h,
                "model": family,
                "feature": name,
                "training_missing_fraction": float(numeric[name].isna().mean()),
                "all_missing_in_training": bool(numeric[name].isna().all()),
            })
        del numeric

        fitted = fit_ridge(
            training_common, y_train, columns,
            alpha=model_settings["ridge_alpha"],
            minimum_rows=model_settings["minimum_fit_rows"],
        )
        train_predictions[family] = predict_ridge(fitted, training_common)
        validation_predictions[family] = predict_ridge(fitted, validation_common)

        saved = save_pipeline(
            fitted, model_dir / f"{family}-{h}s.joblib"
        )
        run_state["files"].append({"kind": "model", **saved})
        fit_records.append({
            "horizon_seconds": h,
            "model": family,
            "training_rows": len(training_common),
            "validation_rows": len(validation_common),
            "training_row_sha256": training_row_hash,
            "validation_row_sha256": validation_row_hash,
            "features": columns,
        })

        names = fitted.named_steps["impute"].get_feature_names_out(columns)
        coefficient_tables.append(pd.DataFrame({
            "horizon_seconds": h,
            "model": family,
            "transformed_feature": names,
            "standardized_coefficient_pips": fitted.named_steps["ridge"].coef_,
        }))

    forecast = validation_common[[
        "feature_timestamp_utc",
        "source_timestamp_utc",
        "snapshot_staleness_seconds",
        "spread_pips",
        f"target_timestamp_utc_{h}s",
        f"label_{h}s_pips",
        f"same_source_record_{h}s",
    ]].rename(columns={
        f"target_timestamp_utc_{h}s": "target_timestamp_utc",
        f"label_{h}s_pips": "actual_change_pips",
        f"same_source_record_{h}s": "same_source_record",
    }).reset_index(drop=True)
    forecast["horizon_seconds"] = h

    spread = forecast["spread_pips"].to_numpy(dtype=float)
    same_source = forecast["same_source_record"].to_numpy(dtype=bool)

    for name, predictions in validation_predictions.items():
        if len(predictions) != len(y_validation) or not np.isfinite(predictions).all():
            raise RuntimeError(f"Invalid matched forecasts: {name}, {h}s")

        forecast[f"prediction_{name}_pips"] = predictions
        metrics = regression_metrics(y_validation, predictions)

        ratio = np.full(len(spread), np.nan)
        np.divide(
            np.abs(predictions), spread,
            out=ratio, where=spread > 0,
        )
        finite_ratio = ratio[np.isfinite(ratio)]
        metric_records.append({
            "horizon_seconds": h,
            "model": name,
            **metrics,
            "median_spread_pips": float(np.median(spread)),
            "median_abs_forecast_pips": float(np.median(np.abs(predictions))),
            "median_abs_forecast_over_spread": (
                float(np.median(finite_ratio)) if len(finite_ratio) else None
            ),
        })

        cuts = training_bucket_cuts(
            train_predictions[name], model_settings["forecast_buckets"]
        )
        bucket_cut_records[f"{h}s/{name}"] = cuts.tolist()
        buckets = forecast_buckets(y_validation, predictions, cuts)
        buckets.insert(0, "model", name)
        buckets.insert(0, "horizon_seconds", h)
        bucket_tables.append(buckets)

        hourly = hourly_errors(
            forecast["feature_timestamp_utc"], y_validation, predictions
        )
        hourly.insert(0, "model", name)
        hourly.insert(0, "horizon_seconds", h)
        hourly_tables.append(hourly)

        for flag, group_name in (
            (True, "same_source_record"),
            (False, "different_source_record"),
        ):
            selected = same_source == flag
            if selected.any():
                conditional_records.append({
                    "horizon_seconds": h,
                    "model": name,
                    "endpoint_group": group_name,
                    **regression_metrics(
                        y_validation[selected], predictions[selected]
                    ),
                })

    forecast_path = forecast_dir / f"validation-{h}s.parquet"
    saved = feature_io._save_frame(forecast, forecast_path)
    run_state["files"].append({
        "kind": "validation_forecasts",
        "path": str(forecast_path),
        **saved,
    })
    run_state["completed_horizons"].append(h)
    feature_io.write_json(run_manifest_path, run_state)

    print(
        f"{h}s complete: "
        f"{len(training_common):,} matched training rows; "
        f"{len(validation_common):,} matched validation rows."
    )

    del fitted, training_common, validation_common, forecast

fit_runtime = time.perf_counter() - fit_started

metrics_table = pd.DataFrame(metric_records)
coverage_table = pd.DataFrame(coverage_records)
bucket_table = pd.concat(bucket_tables, ignore_index=True)
hourly_table = pd.concat(hourly_tables, ignore_index=True)
conditional_table = pd.DataFrame(conditional_records)
coefficient_table = pd.concat(coefficient_tables, ignore_index=True)
missing_feature_table = pd.DataFrame(missing_feature_records)

print("Fitting and diagnostics runtime:", round(fit_runtime, 3), "seconds")

In [ ]:
comparison_records = []

for h in HORIZONS:
    group = metrics_table.loc[metrics_table["horizon_seconds"].eq(h)]
    if set(group["model"]) != {
        "zero", "trailing", "ridge_price", "ridge_price_size"
    }:
        raise RuntimeError("A benchmark or Ridge result is missing.")
    if group["rows"].nunique() != 1:
        raise RuntimeError("Models were evaluated on different observations.")

    fits = [record for record in fit_records if record["horizon_seconds"] == h]
    if (
        len({record["training_row_sha256"] for record in fits}) != 1
        or len({record["validation_row_sha256"] for record in fits}) != 1
    ):
        raise RuntimeError("Ridge feature-family populations differ.")

    by_model = group.set_index("model")
    zero_mae = float(by_model.loc["zero", "mae_pips"])
    zero_rmse = float(by_model.loc["zero", "rmse_pips"])

    for index in group.index:
        metrics_table.loc[index, "mae_gain_vs_zero_pips"] = (
            zero_mae - metrics_table.loc[index, "mae_pips"]
        )
        metrics_table.loc[index, "rmse_gain_vs_zero_pips"] = (
            zero_rmse - metrics_table.loc[index, "rmse_pips"]
        )

    comparison_records.append({
        "horizon_seconds": h,
        "validation_rows": int(group["rows"].iloc[0]),
        "delta_mae_size_minus_price_pips": float(
            by_model.loc["ridge_price_size", "mae_pips"]
            - by_model.loc["ridge_price", "mae_pips"]
        ),
        "delta_rmse_size_minus_price_pips": float(
            by_model.loc["ridge_price_size", "rmse_pips"]
            - by_model.loc["ridge_price", "rmse_pips"]
        ),
    })

comparison_table = pd.DataFrame(comparison_records)

if (
    _sha256(feature_manifest_path) != input_hashes["features"]
    or _sha256(label_manifest_path) != input_hashes["labels"]
):
    raise RuntimeError("An input manifest changed during modeling.")

if not np.isfinite(metrics_table[["mae_pips", "rmse_pips"]]).all().all():
    raise RuntimeError("Nonfinite evaluation errors were produced.")

print("VALIDATION_METRICS")
print(metrics_table[[
    "horizon_seconds", "model", "rows", "mae_pips", "rmse_pips",
    "mae_gain_vs_zero_pips", "rmse_gain_vs_zero_pips",
]].to_string(index=False))

print("\nDISPLAYED_SIZE_COMPARISON — negative differences favor added size")
print(comparison_table.to_string(index=False))

print("\nCOVERAGE")
print(coverage_table[[
    "horizon_seconds", "role", "clock_rows",
    "price_eligible_rows", "matched_rows",
    "price_rows_without_required_size",
]].to_string(index=False))

print("\nPASS: matched populations and unchanged input manifests.")

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

for axis, h in zip(axes.flat, HORIZONS):
    values = []
    for name, label in (
        ("ridge_price", "Price"),
        ("ridge_price_size", "Price + size"),
    ):
        selected = bucket_table.loc[
            bucket_table["horizon_seconds"].eq(h)
            & bucket_table["model"].eq(name)
        ].sort_values("bucket")

        axis.plot(
            selected["mean_forecast_pips"],
            selected["mean_actual_pips"],
            marker="o", label=label,
        )
        values.extend(selected["mean_forecast_pips"].tolist())
        values.extend(selected["mean_actual_pips"].tolist())

    low, high = min(values), max(values)
    padding = max((high - low) * 0.1, 0.001)
    axis.plot(
        [low - padding, high + padding],
        [low - padding, high + padding],
        linestyle="--", color="grey", linewidth=1,
    )
    axis.set_title(f"{h}-second horizon")
    axis.set_xlabel("Mean forecast (pips)")
    axis.set_ylabel("Mean subsequent change (pips)")
    axis.grid(alpha=0.25)
    axis.legend()

figure.suptitle("Development validation: training-defined forecast buckets")

figure_path = (
    REPO_ROOT / "reports/figures" / f"baseline-buckets-{run_id}.png"
)
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=160)
plt.show()
plt.close(figure)

run_state["files"].append({
    "kind": "figure",
    "path": str(figure_path),
    "bytes": figure_path.stat().st_size,
    "sha256": _sha256(figure_path),
})

print("Saved:", figure_path.relative_to(REPO_ROOT).as_posix())

In [ ]:
def relative_path(path):
    return Path(path).resolve().relative_to(REPO_ROOT).as_posix()


def update_section(path, name, content):
    begin = f"<!-- {name}:BEGIN -->"
    finish = f"<!-- {name}:END -->"
    existing = path.read_text(encoding="utf-8") if path.exists() else ""
    if existing.count(begin) != existing.count(finish) or existing.count(begin) > 1:
        raise RuntimeError(f"Malformed generated section: {path.name}")
    block = f"{begin}\n{content.strip()}\n{finish}"
    if begin in existing:
        left, rest = existing.split(begin, 1)
        _, right = rest.split(finish, 1)
        result = left + block + right
    else:
        result = existing.rstrip() + "\n\n" + block + "\n"
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(result)


tables = {
    "validation_metrics": metrics_table,
    "coverage": coverage_table,
    "displayed_size_comparison": comparison_table,
    "forecast_buckets": bucket_table,
    "hourly_errors": hourly_table,
    "endpoint_groups": conditional_table,
    "standardized_coefficients": coefficient_table,
    "training_feature_missingness": missing_feature_table,
}

table_paths = {}
for name, table in tables.items():
    path = table_dir / f"{name}.csv"
    table.to_csv(path, index=False)
    table_paths[name] = relative_path(path)
    run_state["files"].append({
        "kind": "table",
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": _sha256(path),
    })

environment_path = REPO_ROOT / "reports/private/validated_modeling_environment.json"
feature_io.write_json(environment_path, {
    "python": platform.python_version(),
    "packages": versions,
    "validation": test_summary,
})

example_path = REPO_ROOT / "configs/research_config.example.json"
local_path = REPO_ROOT / "configs/research_config.local.json"

example = json.loads(example_path.read_text(encoding="utf-8-sig"))
local = json.loads(local_path.read_text(encoding="utf-8-sig"))

example["baseline_models"] = model_settings
if "baseline_models" in local and local["baseline_models"] != model_settings:
    raise RuntimeError("A conflicting local modeling configuration needs review.")
local.pop("baseline_models", None)
local["active_baseline_manifest"] = relative_path(run_manifest_path)
local["baseline_development_scope"] = windows

feature_io.write_json(example_path, example)
feature_io.write_json(local_path, local)

public_note = f"""
### Notebook 04 — development baselines

Implemented zero-change and trailing-return forecasts, plus training-only
imputation, scaling, and Ridge regression.

Price-only and price-plus-size Ridge models use identical training and
validation observations. Price and size coverage are reported separately.
Alpha is fixed at {model_settings["ridge_alpha"]}; no tuning was performed.

A small development sample is selected from current-price coverage.
The pilot holdout is not loaded or evaluated.

Validation includes MAE, RMSE, descriptive calibration, training-defined
forecast buckets, spread-scale comparisons, hourly diagnostics, and
same-source-record endpoint diagnostics.

Automated execution checks: {test_summary}.
Private execution reports contain the measured development metrics.
Notebook 04 owner review remains pending.

These short-sample comparisons do not establish statistical significance,
full-history stability, or achievable trading profits.

Full walk-forward and final-test research results remain pending execution
and validation.
"""

update_section(REPO_ROOT / "README.md", "NOTEBOOK_04", public_note)
update_section(
    REPO_ROOT / "reports/research_results.md",
    "NOTEBOOK_04",
    public_note,
)
update_section(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK_04",
    f"""
### Notebook 04

- Raymond Fendi requested the baseline-model notebook.
- The Notebook 03 learning checkpoint was discussed and split purging clarified;
  no formal owner code review is claimed.
- OpenAI Codex supplied modeling, evaluation, tests, and documentation.
- Executed correctness checks: {test_summary}.
- Matching evaluation populations and input-manifest integrity checks passed.
- No parameter tuning or pilot holdout evaluation was performed.
- Notebook 04 owner review and understanding checkpoint: pending.
""",
)

# Correct the superseded pending statement without altering historical results.
old_note = "The selected ten-second volatility implementation remains pending."
new_note = (
    "The proposed ten-second-return replacement was superseded; "
    "one-second-return volatility is retained."
)
for relative in (
    "README.md", "CONTRIBUTIONS.md",
    "reports/data_quality.md", "reports/feature_dictionary.md",
):
    path = REPO_ROOT / relative
    if path.exists():
        text = path.read_text(encoding="utf-8")
        if old_note in text:
            with path.open("w", encoding="utf-8", newline="\n") as handle:
                handle.write(text.replace(old_note, new_note))

private_report_path = (
    REPO_ROOT / "reports/private" / f"baseline-{run_id}.md"
)
private_report = (
    "# Development baseline execution\n\n"
    "Short development validation only; pilot holdout untouched.\n\n"
    "## Validation metrics\n\n```\n"
    + metrics_table.to_string(index=False)
    + "\n```\n\n## Displayed-size comparison\n\n```\n"
    + comparison_table.to_string(index=False)
    + "\n```\n\nNegative size-minus-price differences favor added size.\n\n"
    "No uncertainty interval or profitability conclusion is established.\n"
)
private_report_path.parent.mkdir(parents=True, exist_ok=True)
private_report_path.write_text(private_report, encoding="utf-8")

run_state["fit_records"] = fit_records
run_state["training_bucket_cuts_pips"] = bucket_cut_records
run_state["validation"] = {
    "pytest": test_summary,
    "matched_populations": "passed",
    "input_manifest_integrity": "passed",
    "pilot_holdout_evaluated": False,
    "owner_review": "pending",
}
run_state["status"] = "complete"
feature_io.write_json(run_manifest_path, run_state)

print("Saved reports, fitted pipelines, forecasts, and active run configuration.")
print("Run:", run_id)
print("Owner review remains pending.")

In [ ]:
def json_records(frame):
    # pandas converts missing diagnostic values to JSON null.
    return json.loads(frame.to_json(orient="records", date_format="iso"))


all_missing = missing_feature_table.loc[
    missing_feature_table["all_missing_in_training"],
    ["horizon_seconds", "model", "feature"],
]

warnings = [
    "This is a short development pilot, not a full-history evaluation.",
    "The pilot holdout was not loaded or evaluated.",
    "No parameter search was performed; Ridge alpha is fixed.",
    "Comparison metrics use matched size-available observations.",
    "Missing trailing features use training-only imputation.",
    "Same-source-record endpoint groups are evaluation diagnostics, not predictors.",
    "Overlapping labels and serial dependence invalidate independent-row inference.",
    "Statistical uncertainty estimates remain pending broader walk-forward evaluation.",
    "Displayed-size denomination and true side-specific quote ages remain unresolved.",
    "Observed dataframe sizes are not total process peak memory.",
    "Forecast accuracy and spread comparisons do not establish achievable profits.",
]
if len(all_missing):
    warnings.append(
        "Some predictors were entirely missing in training; inspect the "
        "saved training_feature_missingness table."
    )

summary = {
    "notebook": NOTEBOOK,
    "run_id": run_id,
    "python_version": platform.python_version(),
    "package_versions": versions,
    "configuration": {
        "feature_job_id": feature_state["job_id"],
        "label_job_id": label_state["job_id"],
        "development_windows": windows,
        "horizons_seconds": HORIZONS,
        "maximum_snapshot_age_seconds": feature_settings["max_staleness_seconds"],
        "volatility_return_interval_seconds": 1,
        "ridge_alpha": model_settings["ridge_alpha"],
        "ridge_solver": "cholesky",
        "parameter_search": "none",
        "price_predictor_count": len(PRICE_FEATURES),
        "added_size_predictor_count": len(SIZE_FEATURES),
        "forecast_buckets_requested": model_settings["forecast_buckets"],
    },
    "inputs": {
        "training_clock_rows": len(frames["train"]),
        "validation_clock_rows": len(frames["validation"]),
        "holdout_rows_loaded": 0,
    },
    "coverage": json_records(coverage_table),
    "validation_metrics": json_records(metrics_table),
    "displayed_size_comparison": json_records(comparison_table),
    "all_missing_training_predictors": json_records(all_missing),
    "validation": {
        "imports_from_notebooks_directory": "passed",
        "pytest": test_summary,
        "selected_input_integrity": "passed",
        "feature_label_alignment": "passed",
        "matched_training_and_validation_populations": "passed",
        "input_manifests_unchanged": "passed",
        "training_only_preprocessing": "implemented and covered by tests",
        "pilot_holdout_evaluated": False,
        "uncertainty_intervals": "deferred",
        "owner_review": "pending",
    },
    "measurements": {
        "fitting_and_diagnostics_seconds": round(fit_runtime, 3),
        "training_frame_bytes": frame_bytes["train"],
        "validation_frame_bytes": frame_bytes["validation"],
        "saved_model_count": sum(
            item["kind"] == "model" for item in run_state["files"]
        ),
        "model_bytes": sum(
            item["bytes"] for item in run_state["files"]
            if item["kind"] == "model"
        ),
        "validation_forecast_rows": sum(
            item["rows"] for item in run_state["files"]
            if item["kind"] == "validation_forecasts"
        ),
        "validation_forecast_parquet_bytes": sum(
            item["bytes"] for item in run_state["files"]
            if item["kind"] == "validation_forecasts"
        ),
        "notebook_elapsed_seconds": round(
            time.perf_counter() - notebook_started, 3
        ),
    },
    "warnings": warnings,
    "saved_outputs": {
        "active_baseline_manifest": relative_path(run_manifest_path),
        "model_directory": relative_path(model_dir),
        "forecast_directory": relative_path(forecast_dir),
        "tables": table_paths,
        "forecast_bucket_figure": relative_path(figure_path),
        "private_report": relative_path(private_report_path),
        "validated_environment": relative_path(environment_path),
        "execution_summary": "reports/private/notebook04_execution.json",
    },
}

feature_io.write_json(
    REPO_ROOT / "reports/private/notebook04_execution.json", summary
)

print("NOTEBOOK_04_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))